# Model Training & Evaluation

Train Random Forest, XGBoost, and LightGBM on **dual targets** (occupancy + volume).  
Load tuned hyperparameters from Notebook 04. Evaluate with MAE, RMSE, R², and MAPE.

## Import Required Libraries


In [1]:
import pandas as pd
import numpy as np
import joblib
import json
import os
import time

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor

import warnings
warnings.filterwarnings('ignore')

print("Libraries loaded!")

Libraries loaded!


## 1. Load Dataset

Load the featured dataset and concatenate.

In [2]:
import glob

parts = sorted(glob.glob("../data/processed/final_featured_dataset_part_*.csv"))
print(f"Found {len(parts)} dataset parts")

df = pd.concat([pd.read_csv(f) for f in parts], ignore_index=True)
df['time'] = pd.to_datetime(df['time'])
df = df.sort_values(['zone_id', 'time']).reset_index(drop=True)

print(f"Dataset shape: {df.shape}")
print(f"Date range: {df['time'].min()} → {df['time'].max()}")
print(f"Zones: {df['zone_id'].nunique()}")
df.head()

Found 4 dataset parts
Dataset shape: (1148400, 39)
Date range: 2022-09-08 00:00:00 → 2023-02-28 23:00:00
Zones: 275


,time,zone_id,occupancy,volume,duration,e_price,s_price,longitude,latitude,charge_count,...,occ_lag_6h,occ_lag_12h,occ_lag_24h,occ_lag_168h,vol_lag_24h,occ_rmean_6h,occ_rmean_12h,occ_rmean_24h,occ_rstd_24h,occ_diff_1h
0,2022-09-08 00:00:00,102,19.0,66.5,10.0,0.924,0.856,114.102962,22.540413,30,...,19.0,19.0,19.0,17.0,66.5,19.0,19.0,19.0,0.0,0.0
1,2022-09-08 01:00:00,102,19.0,66.5,10.0,0.924,0.856,114.102962,22.540413,30,...,19.0,19.0,19.0,16.0,66.5,19.0,19.0,19.0,0.0,0.0
2,2022-09-08 02:00:00,102,19.0,66.5,10.0,0.924,0.856,114.102962,22.540413,30,...,19.0,19.0,19.0,16.0,66.5,19.0,19.0,19.0,0.0,0.0
3,2022-09-08 03:00:00,102,19.0,66.5,10.0,0.924,0.856,114.102962,22.540413,30,...,19.0,19.0,19.0,16.0,66.5,19.0,19.0,19.0,0.0,0.0
4,2022-09-08 04:00:00,102,19.0,66.5,10.0,0.924,0.856,114.102962,22.540413,30,...,19.0,19.0,19.0,17.0,66.5,19.0,19.0,19.0,0.0,0.0


## 2. Load Tuned Hyperparameters

Load best parameters from grid search results. Falls back to sensible defaults if the file doesn't exist.

In [3]:
params_path = "../results/best_hyperparameters.json"

if os.path.exists(params_path):
    with open(params_path, 'r') as f:
        best_params = json.load(f)
    print("Loaded tuned hyperparameters:")
else:
    best_params = {
        "RandomForest": {"n_estimators": 200, "max_depth": 15, "min_samples_leaf": 5},
        "XGBoost": {"learning_rate": 0.05, "max_depth": 5, "n_estimators": 800},
        "LightGBM": {"learning_rate": 0.1, "num_leaves": 31, "n_estimators": 800}
    }
    print("Using default hyperparameters (no JSON found):")

for model, params in best_params.items():
    print(f"  {model}: {params}")

Loaded tuned hyperparameters:
  RandomForest: {'max_depth': 15, 'min_samples_leaf': 5, 'n_estimators': 200}
  XGBoost: {'learning_rate': 0.05, 'max_depth': 5, 'n_estimators': 800}
  LightGBM: {'learning_rate': 0.1, 'n_estimators': 800, 'num_leaves': 31}


## 3. Train/Test Split (Time-Based)

**Sep 2022 – Jan 2023 → Training** | **Feb 2023 → Testing**  
This prevents data leakage and mirrors the article's temporal split approach.

In [ ]:
split_date = pd.Timestamp("2023-02-01")

train = df[df['time'] < split_date].copy()
test  = df[df['time'] >= split_date].copy()

print("Train shape:", train.shape)
print("Test shape:", test.shape)
print(f"Split ratio: {len(train)/(len(train)+len(test))*100:.1f}% / {len(test)/(len(train)+len(test))*100:.1f}%")

# Keep time and zone_id for predictions file; drop from features
meta_cols = ['time', 'zone_id']
target_cols = ['occupancy', 'volume']
drop_cols = meta_cols + target_cols + ['duration', 'e_price', 's_price']

feature_cols = [c for c in df.columns if c not in drop_cols]

print(f"\nFeatures ({len(feature_cols)}): {feature_cols}")

Train shape: (963600, 39)
Test shape: (184800, 39)
Split ratio: 83.9% / 16.1%

Features (32): ['longitude', 'latitude', 'charge_count', 'area', 'perimeter', 'num_stations', 'total_piles', 'mean_station_lat', 'mean_station_lon', 'hour', 'day_of_week', 'month', 'day_of_month', 'is_weekend', 'hour_sin', 'hour_cos', 'dow_sin', 'dow_cos', 'charge_density', 'total_price', 'occ_lag_1h', 'occ_lag_3h', 'occ_lag_6h', 'occ_lag_12h', 'occ_lag_24h', 'occ_lag_168h', 'vol_lag_24h', 'occ_rmean_6h', 'occ_rmean_12h', 'occ_rmean_24h', 'occ_rstd_24h', 'occ_diff_1h']


## 4. Define Evaluation Metrics

Using 4 metrics: **MAE**, **RMSE**, **R²**, and **MAPE** (matching the article's evaluation framework).

In [5]:
def mean_absolute_percentage_error(y_true, y_pred):
    """MAPE — skip zeros in denominator to avoid division errors."""
    mask = y_true != 0
    if mask.sum() == 0:
        return np.nan
    return np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100


def evaluate_model(y_true, y_pred, label=""):
    """Return dict of all 4 metrics."""
    mae  = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2   = r2_score(y_true, y_pred)
    mape = mean_absolute_percentage_error(y_true.values, y_pred)
    return {"MAE": mae, "RMSE": rmse, "R²": r2, "MAPE (%)": mape}


print("Evaluation functions defined")

Evaluation functions defined


## 5. Load Model Configuration

In [ ]:
model_configs = {
    "RandomForest": (
        RandomForestRegressor,
        {
            **best_params["RandomForest"], 
            "n_jobs": -1, 
            "random_state": 42
        }
    ),
    "XGBoost": (
        XGBRegressor,
        {
            **best_params["XGBoost"], 
            "tree_method": "hist", 
            "random_state": 42, 
            "verbosity": 0
        }
    ),
    "LightGBM": (
        LGBMRegressor,
        {
            **best_params["LightGBM"], 
            "n_jobs": -1, 
            "random_state": 42, 
            "verbose": -1
        }
    )
}

## 6. Train Models - Primary Target: Occupancy

Training all 3 models on **occupancy** as a primary prediction target.

### 5.1 Target Split

In [ ]:
X_train = train[feature_cols]
y_train_occ = train['occupancy']
X_test = test[feature_cols]
y_test_occ = test['occupancy']

### 5.2 Training Function

In [7]:
def train_and_evaluate(name, model_class, params):
    print(f"Training {name} (occupancy)...")
    
    model = model_class(**params)
    
    start = time.time()
    model.fit(X_train, y_train_occ)
    elapsed = time.time() - start
    
    results = {
        "train": evaluate_model(y_train_occ, model.predict(X_train)),
        "test":  evaluate_model(y_test_occ, model.predict(X_test)),
        "time":  elapsed
    }
    
    print(f"  Done in {elapsed:.1f}s")
    
    return model, results

### 5.3 Training Loop

In [8]:
models_occ = {}
results_occ = {}

for name, (model_class, params) in model_configs.items():
    model, results = train_and_evaluate(name, model_class, params)
    models_occ[name] = model
    results_occ[name] = results

print("\nAll occupancy models trained!")

Training RandomForest (occupancy)...
  Done in 235.5s
Training XGBoost (occupancy)...
  Done in 8.6s
Training LightGBM (occupancy)...
  Done in 9.7s

All occupancy models trained!


### 5.4 Occupancy results table

In [9]:
print("OCCUPANCY — Test Set Results\n")

rows = []
for name, res in results_occ.items():
    row = {'Model': name, **res['test'], 'Train Time (s)': res['time']}
    rows.append(row)

occ_metrics_df = pd.DataFrame(rows).set_index('Model')
print(occ_metrics_df.to_string())

print("\nOCCUPANCY — Train vs Test (overfitting check)\n")
for name, res in results_occ.items():
    train_r2 = res['train']['R²']
    test_r2  = res['test']['R²']
    gap = train_r2 - test_r2
    print(f"  {name:15s}  Train R²={train_r2:.4f}  Test R²={test_r2:.4f}  Gap={gap:.4f}")

OCCUPANCY — Test Set Results

                   MAE      RMSE        R²  MAPE (%)  Train Time (s)
Model                                                               
RandomForest  0.007457  0.190994  0.999915  0.020309      235.478110
XGBoost       0.157386  0.899407  0.998120  0.621107        8.615578
LightGBM      0.055315  0.141735  0.999953  0.367896        9.727306

OCCUPANCY — Train vs Test (overfitting check)

  RandomForest     Train R²=0.9997  Test R²=0.9999  Gap=-0.0003
  XGBoost          Train R²=0.9981  Test R²=0.9981  Gap=0.0000
  LightGBM         Train R²=1.0000  Test R²=1.0000  Gap=0.0000


## 7. Train Models - Secondary Target: Volume

Training all 3 models on **volume** (kWh) as a secondary prediction target.

### 6.1 Target Split

In [10]:
y_train_vol = train['volume']
y_test_vol  = test['volume']

### 6.2 Training Function

In [11]:
def train_and_evaluate(name, model_class, params, y_train, y_test):
    print(f"Training {name}...")

    model = model_class(**params)

    start = time.time()
    model.fit(X_train, y_train)
    elapsed = time.time() - start

    results = {
        "train": evaluate_model(y_train, model.predict(X_train)),
        "test":  evaluate_model(y_test, model.predict(X_test)),
        "time":  elapsed
    }

    print(f"  Done in {elapsed:.1f}s")
    return model, results

### 6.3 Training Loop

In [12]:
models_vol = {}
results_vol = {}

for name, (model_class, params) in model_configs.items():
    model, results = train_and_evaluate(
        name=f"{name} (volume)",
        model_class=model_class,
        params=params,
        y_train=y_train_vol,
        y_test=y_test_vol,
    )

    models_vol[name] = model
    results_vol[name] = results

print("\nAll volume models trained!")

Training RandomForest (volume)...
  Done in 292.2s
Training XGBoost (volume)...
  Done in 9.0s
Training LightGBM (volume)...
  Done in 8.1s

All volume models trained!


### 6.4 Volume results table

In [13]:
print("VOLUME — Test Set Results\n")

rows = []
for name, res in results_vol.items():
    row = {'Model': name, **res['test'], 'Train Time (s)': res['time']}
    rows.append(row)

vol_metrics_df = pd.DataFrame(rows).set_index('Model')
print(vol_metrics_df.to_string())

print("\nVOLUME — Train vs Test (overfitting check)\n")
for name, res in results_vol.items():
    train_r2 = res['train']['R²']
    test_r2  = res['test']['R²']
    gap = train_r2 - test_r2
    print(f"  {name:15s}  Train R²={train_r2:.4f}  Test R²={test_r2:.4f}  Gap={gap:.4f}")

VOLUME — Test Set Results

                    MAE        RMSE        R²   MAPE (%)  Train Time (s)
Model                                                                   
RandomForest  46.309226  181.381732  0.954743  43.887091      292.217120
XGBoost       44.255036  169.637343  0.960414  45.871004        8.963988
LightGBM      43.844577  170.664440  0.959933  48.777070        8.079681

VOLUME — Train vs Test (overfitting check)

  RandomForest     Train R²=0.9782  Test R²=0.9547  Gap=0.0234
  XGBoost          Train R²=0.9716  Test R²=0.9604  Gap=0.0112
  LightGBM         Train R²=0.9836  Test R²=0.9599  Gap=0.0237


## 8. Save Models, Predictions, Metrics & Feature Importance

### 8.1 Save Models

In [14]:
def save_models(models_dict, target, base_dir="../models"):
    os.makedirs(base_dir, exist_ok=True)

    for name, model in models_dict.items():
        path = f"{base_dir}/{name.lower()}_{target}.pkl"
        joblib.dump(model, path)
        print(f"Saved {path}")

save_models(models_occ, "occupancy")
save_models(models_vol, "volume")
print("\nAll models saved!")

Saved ../models/randomforest_occupancy.pkl
Saved ../models/xgboost_occupancy.pkl
Saved ../models/lightgbm_occupancy.pkl
Saved ../models/randomforest_volume.pkl
Saved ../models/xgboost_volume.pkl
Saved ../models/lightgbm_volume.pkl

All models saved!


### 8.2 Save Predictions

In [15]:
def save_predictions(path="../results/predictions/test_predictions.csv"):
    os.makedirs(os.path.dirname(path), exist_ok=True)

    pred_data = {
        "time": test["time"].values,
        "zone_id": test["zone_id"].values,
        "actual_occupancy": y_test_occ.values,
        "actual_volume": y_test_vol.values,
    }

    # Add model predictions dynamically
    for name in models_occ:
        pred_data[f"{name.lower()}_occ_pred"] = models_occ[name].predict(X_test)
        pred_data[f"{name.lower()}_vol_pred"] = models_vol[name].predict(X_test)

    pred_df = pd.DataFrame(pred_data)
    pred_df.to_csv(path, index=False)

    print(f"\nPredictions saved: {pred_df.shape}")
    return pred_df

pred_df = save_predictions()


Predictions saved: (184800, 10)


### 8.3 Save Metrics

In [16]:
def save_metrics(path="../results/metrics/model_metrics.csv"):
    os.makedirs(os.path.dirname(path), exist_ok=True)

    all_metrics = []

    for target, results_dict in {
        "occupancy": results_occ,
        "volume": results_vol,
    }.items():
        for name, res in results_dict.items():
            all_metrics.append({
                "target": target,
                "model": name,
                **res["test"],
                "train_time_s": res["time"],
            })

    metrics_df = pd.DataFrame(all_metrics)
    metrics_df.to_csv(path, index=False)

    print(f"Metrics saved: {metrics_df.shape}")
    print(metrics_df.to_string(index=False))

    return metrics_df

metrics_df = save_metrics()

Metrics saved: (6, 7)
   target        model       MAE       RMSE       R²  MAPE (%)  train_time_s
occupancy RandomForest  0.007457   0.190994 0.999915  0.020309    235.478110
occupancy      XGBoost  0.157386   0.899407 0.998120  0.621107      8.615578
occupancy     LightGBM  0.055315   0.141735 0.999953  0.367896      9.727306
   volume RandomForest 46.309226 181.381732 0.954743 43.887091    292.217120
   volume      XGBoost 44.255036 169.637343 0.960414 45.871004      8.963988
   volume     LightGBM 43.844577 170.664440 0.959933 48.777070      8.079681


### 8.4 Save Feature Importance (All Targets)

In [17]:
def save_feature_importance(models_dict, target, base_dir="../results/feature_importance"):
    os.makedirs(base_dir, exist_ok=True)

    for name, model in models_dict.items():
        if hasattr(model, "feature_importances_"):
            fi_df = pd.DataFrame({
                "feature": feature_cols,
                "importance": model.feature_importances_,
            }).sort_values("importance", ascending=False)

            path = f"{base_dir}/{name.lower()}_{target}_feature_importance.csv"
            fi_df.to_csv(path, index=False)

            print(f"Feature importance saved: {path}")

save_feature_importance(models_occ, "occupancy")
save_feature_importance(models_vol, "volume")

print("\n✓ All files saved successfully!")

Feature importance saved: ../results/feature_importance/randomforest_occupancy_feature_importance.csv
Feature importance saved: ../results/feature_importance/xgboost_occupancy_feature_importance.csv
Feature importance saved: ../results/feature_importance/lightgbm_occupancy_feature_importance.csv
Feature importance saved: ../results/feature_importance/randomforest_volume_feature_importance.csv
Feature importance saved: ../results/feature_importance/xgboost_volume_feature_importance.csv
Feature importance saved: ../results/feature_importance/lightgbm_volume_feature_importance.csv

✓ All files saved successfully!
